#### **Feature Engineering & Financial Metrics**

This notebook implements feature engineering logic based on approved design rules
from Task 3.1.

Input:
- equity_df_std (cleaned & standardized dataset)

Output:
- equity_features_df (feature-enriched dataset)


In [12]:
# -------------------------------------------
# Notebook bootstrap (LOCKED)
# -------------------------------------------

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("SRC path added:", SRC_PATH)

SRC path added: c:\Users\dhira\Desktop\equity-analytics\equity-fundamentals-analytics\src


In [13]:
from equity_analytics.io import load_equity_df_std

equity_df_std = load_equity_df_std()
print("Loaded shape:", equity_df_std.shape)

equity_features_df = equity_df_std.copy()
equity_features_df.head()


Loaded shape: (3145, 19)


,s_no,industry,company_name,cmp_rs,market_cap_cr,debt_to_eq,sales_cr,sales_growth_pct,sales_qtr_cr,profit_growth_pct,profit_var_5yrs_pct,roe_pct,opm_pct,eps_12m_rs,peg,pe,pat_12m_cr,roce_pct,cmp_to_bv
0,1,Auto Components Companies,Samvardh. Mothe.,112.16,118389.50,0.53,117367.72,7.30,30172.97,-11.59,26.02,12.16,8.86,3.10,0.59,35.35,3348.88,13.66,3.19
1,2,Auto Components Companies,Bosch,36870.00,108780.78,0.01,18959.70,10.55,4794.80,13.42,12.76,15.55,13.11,909.18,2.61,47.89,2271.53,21.11,7.76
2,3,Auto Components Companies,Uno Minda,1305.10,75253.94,0.46,18015.43,17.14,4814.03,21.44,40.73,17.50,11.54,19.02,1.74,68.82,1093.46,18.83,12.08
3,4,Auto Components Companies,Bharat Forge,1446.20,69113.96,0.71,15268.83,-3.52,4031.93,3.89,20.51,11.58,17.66,22.54,80.94,63.94,1080.93,12.18,7.39
4,5,Auto Components Companies,MRF,153355.00,65066.94,0.19,29130.05,9.56,7378.72,-4.82,5.75,10.62,14.11,4370.73,0.70,35.10,1853.67,13.62,3.35


#### **Valuation Features**

In [14]:
# -------------------------------------------
# Feature Group 1: Valuation Features
# -------------------------------------------

# 1) is_valuation_known
# True only when both PE and CMP/BV are available (not missing)

equity_features_df["is_valuation_known"] = (
    equity_features_df["pe"].notna() & equity_features_df["cmp_to_bv"].notna())

# 2) valuation_bucket
# Business rules (approved):
# - unknown if PE or CMP/BV is missing
# - cheap: PE < 15 AND CMP/BV < 1.5
# - fair:  (15 <= PE <= 30) OR (1.5 <= CMP/BV <= 3)
# - expensive: PE > 30 OR CMP/BV > 3

import numpy as np

# -----------------------------
# Step 1: Define masks for valuation categories
# -----------------------------

# Mask for stocks where valuation data is unknown
unknown_mask = ~equity_features_df["is_valuation_known"]

# Mask for "cheap" stocks:
# PE ratio < 15 AND Price-to-Book ratio < 1.5
cheap_mask = (
    equity_features_df["pe"].lt(15) & equity_features_df["cmp_to_bv"].lt(1.5)
)

# Mask for "fairly valued" stocks:
# PE ratio between 15 and 30 OR Price-to-Book ratio between 1.5 and 3
fair_mask = (
    equity_features_df['pe'].between(15, 30, inclusive="both") | 
    equity_features_df["cmp_to_bv"].between(1.5, 3, inclusive="both")
)

# Mask for "expensive" stocks:
# PE ratio > 30 OR Price-to-Book ratio > 3
expensive_mask = (
    equity_features_df["pe"].gt(30) | equity_features_df["cmp_to_bv"].gt(3)
)

# -----------------------------
# Step 2: Assign valuation buckets based on the masks
# -----------------------------
# np.select checks the masks in order and assigns the corresponding label.
# Default is "fair" if none of the masks match or if there are overlaps.
equity_features_df["valuation_bucket"] = np.select(
    condlist=[unknown_mask, cheap_mask, fair_mask, expensive_mask],
    choicelist=["unknown", "cheap", "fair", "expensive"],
    default="fair",  # safe default when conditions overlap oddly
)

# -----------------------------
# Step 3: Check for missing values
# -----------------------------
print("Null check:")
print(equity_features_df[["is_valuation_known","valuation_bucket"]].isna().sum())

# -----------------------------
# Step 4: View distribution of valuation buckets
# -----------------------------
print("\nValue counts:")
print(equity_features_df["valuation_bucket"].value_counts(dropna=False))

# -----------------------------
# Step 5: Sample view of the data
# -----------------------------
print("\nSample view:")
print(
    equity_features_df[
        ["company_name", "pe","cmp_to_bv","is_valuation_known","valuation_bucket"]
    ].head(10)
)

Null check:
is_valuation_known    0
valuation_bucket      0
dtype: int64

Value counts:
valuation_bucket
fair         1190
expensive     916
unknown       714
cheap         325
Name: count, dtype: int64

Sample view:
       company_name     pe  cmp_to_bv  is_valuation_known valuation_bucket
0  Samvardh. Mothe.  35.35       3.19                True        expensive
1             Bosch  47.89       7.76                True        expensive
2         Uno Minda  68.82      12.08                True        expensive
3      Bharat Forge  63.94       7.39                True        expensive
4               MRF  35.10       3.35                True        expensive
5  Schaeffler India  57.46      11.66                True        expensive
6  Tube Investments  93.70       7.81                True        expensive
7   Balkrishna Inds  32.48       4.29                True        expensive
8   Endurance Tech.  44.03       6.15                True        expensive
9      Apollo Tyres  25.31       

#### **Profitability & Efficiency**

In [15]:
# -------------------------------------------
# Feature Group 2: Profitability & Efficiency
# -------------------------------------------

# -----------------------------
# 1) is_profitable
# -----------------------------
# Business rule:
# - A company is considered profitable if:
#   ROE >= 10 AND OPM >= 10
equity_features_df["is_profitable"] = (
    equity_features_df['roe_pct'].ge(10)
    & equity_features_df["opm_pct"].ge(10)
)

# -----------------------------
# 2) profitability_bucket
# -----------------------------
# Business interpretation:
# - weak: ROE < 10 OR OPM < 10
# - moderate: ROE between 10–20 AND OPM between 10–20
# - strong: ROE > 20 AND OPM > 20

import numpy as np

# Mask for weak profitability
weak_mask = (
    (equity_features_df['roe_pct'] < 10) | 
    (equity_features_df['opm_pct'] < 10)
)

# Mask for moderate profitability
moderate_mask = (
    equity_features_df['roe_pct'].between(10, 20, inclusive="both") & 
    equity_features_df["opm_pct"].between(10, 20, inclusive="both")
)

# Mask for strong profitability
strong_mask = (
    equity_features_df['roe_pct'].gt(20) & 
    equity_features_df['opm_pct'].gt(20)
)

# Assign profitability bucket based on masks
# Default is 'moderate' if none of the masks match
equity_features_df['profitability_bucket'] = np.select(
    condlist=[weak_mask, moderate_mask, strong_mask],
    choicelist=["weak", "moderate", "strong"],
    default='moderate'
)

# -----------------------------
# 3) capital_efficiency_flag
# -----------------------------
# ROCE > 15 indicates efficient use of capital
equity_features_df['capital_efficiency_flag'] = (
    equity_features_df["roce_pct"].gt(15)
)

# -----------------------------
# 4) Data quality checks and insights
# -----------------------------
# Check for any missing values in the newly created features
print("Null check:")
print(
    equity_features_df[
        ["is_profitable", "profitability_bucket", "capital_efficiency_flag"]
    ].isna().sum()
)

# View the distribution of profitability buckets
print("\nProfitability bucket distribution:")
print(equity_features_df["profitability_bucket"].value_counts())

# Sample view of the key columns for verification
print("\nSample view:")
print(
    equity_features_df[
        [
            "company_name",
            "roe_pct",
            "opm_pct",
            "roce_pct",
            "is_profitable",
            "profitability_bucket",
            "capital_efficiency_flag",
        ]
    ].head(10)
)

Null check:
is_profitable              0
profitability_bucket       0
capital_efficiency_flag    0
dtype: int64

Profitability bucket distribution:
profitability_bucket
weak        1937
moderate     969
strong       239
Name: count, dtype: int64

Sample view:
       company_name  roe_pct  opm_pct  roce_pct  is_profitable  \
0  Samvardh. Mothe.    12.16     8.86     13.66          False   
1             Bosch    15.55    13.11     21.11           True   
2         Uno Minda    17.50    11.54     18.83           True   
3      Bharat Forge    11.58    17.66     12.18           True   
4               MRF    10.62    14.11     13.62           True   
5  Schaeffler India    19.17    18.98     25.67           True   
6  Tube Investments    12.77     9.22     21.80          False   
7   Balkrishna Inds    15.78    21.32     16.67           True   
8   Endurance Tech.    14.63    13.48     17.26           True   
9      Apollo Tyres     8.61    13.73     11.44          False   

  profitabili

**Growth Signal Features**

In [16]:
# -------------------------------------------
# Feature Group 3: Growth Signals
# -------------------------------------------

# -----------------------------
# 1) is_growth_company
# -----------------------------
# Business rule:
# - Sales growth >= 10%
# - Profit growth >= 10%
# - 5-year profit trend should be positive
#
# If all three conditions are satisfied,
# the company is considered a growth company
equity_features_df["is_growth_company"] = (
    equity_features_df["sales_growth_pct"].ge(10) & 
    equity_features_df["profit_growth_pct"].ge(10) &
    equity_features_df["profit_var_5yrs_pct"].ge(0)
)

# -----------------------------
# 2) growth_bucket
# -----------------------------
# Interpretation:
# - weak: sales OR profit growth < 10
# - moderate: sales & profit growth between 10–20
# - strong: sales & profit growth > 20 AND positive long-term trend

import numpy as np

# Mask for weak growth:
# Either sales growth or profit growth is below 10%
weak_growth = (
    (equity_features_df["sales_growth_pct"] < 10) |
    (equity_features_df["profit_growth_pct"] < 10)
)

# Mask for moderate growth:
# Both sales growth and profit growth are between 10% and 20%
moderate_growth = (
    equity_features_df['sales_growth_pct'].between(10, 20, inclusive="both") &
    equity_features_df["profit_growth_pct"].between(10, 20, inclusive="both")
)

# Mask for strong growth:
# Sales growth > 20%, profit growth > 20%,
# and positive 5-year profit trend
strong_growth = (
    equity_features_df["sales_growth_pct"].gt(20) &
    equity_features_df["profit_growth_pct"].gt(20) &
    equity_features_df["profit_var_5yrs_pct"].gt(0)
)

# Assign growth bucket using numpy select
# Default is set to "moderate" as a safe fallback
equity_features_df["growth_bucket"] = np.select(
    condlist=[weak_growth, moderate_growth, strong_growth],
    choicelist=["weak", "moderate", "strong"],
    default="moderate"
)

# -------------------------------------------
# Validations
# -------------------------------------------

# Check for missing values in growth-related features
print("Null check:")
print(
    equity_features_df[
        ["is_growth_company", "growth_bucket"]
    ].isna().sum()
)

# View distribution of growth buckets
print("\nGrowth bucket distribution:")
print(equity_features_df["growth_bucket"].value_counts())

# Sample view for manual verification
print("\nSample view:")
equity_features_df[
    [
        "company_name",
        "sales_growth_pct",
        "profit_growth_pct",
        "profit_var_5yrs_pct",
        "is_growth_company",
        "growth_bucket",
    ]
].head(10)

Null check:
is_growth_company    0
growth_bucket        0
dtype: int64

Growth bucket distribution:
growth_bucket
weak        1979
moderate     750
strong       416
Name: count, dtype: int64

Sample view:


,company_name,sales_growth_pct,profit_growth_pct,profit_var_5yrs_pct,is_growth_company,growth_bucket
0,Samvardh. Mothe.,7.30,-11.59,26.02,False,weak
1,Bosch,10.55,13.42,12.76,True,moderate
2,Uno Minda,17.14,21.44,40.73,True,moderate
3,Bharat Forge,-3.52,3.89,20.51,False,weak
4,MRF,9.56,-4.82,5.75,False,weak
5,Schaeffler India,12.56,17.75,21.59,True,moderate
6,Tube Investments,14.32,-22.38,16.40,False,weak
7,Balkrishna Inds,3.72,-15.32,12.99,False,weak
8,Endurance Tech.,15.84,13.57,7.19,True,moderate
9,Apollo Tyres,4.36,-13.38,21.05,False,weak


#### **Final Sanity Checks + Save Feature Dataset**

In [17]:
# -------------------------------------------
# Final sanity checks + persist features dataset
# -------------------------------------------

from equity_analytics.io import save_equity_df_std
from pathlib import Path

# 1) Sanity checks: required feature columns must exist
required_features = [
    "is_valuation_known", "valuation_bucket",
    "is_profitable", "profitability_bucket", "capital_efficiency_flag",
    "is_growth_company", "growth_bucket"
]

missing_cols = [c for c in required_features if c not in equity_features_df.columns]
if missing_cols:
    raise ValueError(f"Missing expected feature columns: {missing_cols}")

print("✅ All expected feature columns present.")

# 2) Quick distribution checks (helps spot weird results)
print("\nValuation bucket counts:")
print(equity_features_df["valuation_bucket"].value_counts(dropna=False))

print("\nProfitability bucket counts:")
print(equity_features_df["profitability_bucket"].value_counts(dropna=False))

print("\nGrowth bucket counts:")
print(equity_features_df["growth_bucket"].value_counts(dropna=False))

# 3) Save features dataset (new file)
processed_dir = Path("data") / "processed"
features_path = processed_dir / "equity_features_df.csv"
processed_dir.mkdir(parents=True, exist_ok=True)

equity_features_df.to_csv(features_path, index=False)
print(f"\n✅ Saved features dataset to: {features_path}")


✅ All expected feature columns present.

Valuation bucket counts:
valuation_bucket
fair         1190
expensive     916
unknown       714
cheap         325
Name: count, dtype: int64

Profitability bucket counts:
profitability_bucket
weak        1937
moderate     969
strong       239
Name: count, dtype: int64

Growth bucket counts:
growth_bucket
weak        1979
moderate     750
strong       416
Name: count, dtype: int64

✅ Saved features dataset to: data\processed\equity_features_df.csv


#### **Screening Logic — Basic Screen**

In [18]:
# -------------------------------------------
# Screening Logic — Basic Screen
# -------------------------------------------

# Business rule:
# A company passes the basic screen if it satisfies the following conditions:
# 1. The company's valuation is known.
# 2. The company is profitable (i.e., has positive profits).
# 3. The company's capital efficiency is considered acceptable.

# We create a new column `is_basic_screen_pass` in the `equity_features_df` DataFrame
# that will hold a boolean value (True or False) depending on whether the company 
# meets the conditions for passing the basic screen.

# The following conditions are checked:
# - `is_valuation_known`: If the company's valuation is available (True if known, False if unknown).
# - `is_profitable`: If the company is profitable (True if profitable, False if not).
# - `capital_efficiency_flag`: If the company's capital efficiency is acceptable (True if acceptable, False if not).

# The result is the logical AND operation across all three conditions.
# If all three conditions are True, the company passes the basic screen (is_basic_screen_pass = True).
# If any condition is False, the company fails the basic screen (is_basic_screen_pass = False).

equity_features_df["is_basic_screen_pass"] = (
    equity_features_df["is_valuation_known"] &  # Check if valuation is known
    equity_features_df["is_profitable"] &       # Check if company is profitable
    equity_features_df["capital_efficiency_flag"]  # Check if capital efficiency is acceptable
)

# -------------------------------------------
# Validations
# -------------------------------------------

print("Null Check:")
print(equity_features_df["is_basic_screen_pass"].isna().sum())

print("\nPass / Fail Counts:")
print(equity_features_df["is_basic_screen_pass"].value_counts())

print("\nSample view:")
equity_features_df[
    [
        "company_name",
        "is_valuation_known",
        "is_profitable",
        "capital_efficiency_flag",
        "is_basic_screen_pass"
    ]
].head(10)

Null Check:
0

Pass / Fail Counts:
is_basic_screen_pass
False    2324
True      821
Name: count, dtype: int64

Sample view:


,company_name,is_valuation_known,is_profitable,capital_efficiency_flag,is_basic_screen_pass
0,Samvardh. Mothe.,True,False,False,False
1,Bosch,True,True,True,True
2,Uno Minda,True,True,True,True
3,Bharat Forge,True,True,False,False
4,MRF,True,True,False,False
5,Schaeffler India,True,True,True,True
6,Tube Investments,True,False,True,False
7,Balkrishna Inds,True,True,True,True
8,Endurance Tech.,True,True,True,True
9,Apollo Tyres,True,False,False,False


#### **Screening Logic — Quality Screen**

In [19]:
# -------------------------------------------
# Screening Logic — Quality Screen
# -------------------------------------------

# Business rule:
# A company passes the quality screen if:
# - The profitability bucket is "strong"
# - The capital efficiency is acceptable (i.e., capital_efficiency_flag is True)

# The following line creates a new column 'is_quality_screen_pass' that evaluates to True 
# if both conditions are met for each company in the dataframe. 
# It checks if the 'profitability_bucket' is "strong" and the 'capital_efficiency_flag' is True.
equity_features_df["is_quality_screen_pass"] = (
    equity_features_df["profitability_bucket"].eq("strong")  # Check if profitability bucket is 'strong'
    & equity_features_df["capital_efficiency_flag"]  # Check if capital efficiency is acceptable
)

# -------------------------------------------
# Validations
# -------------------------------------------

# Checking for any missing values (nulls) in the 'is_quality_screen_pass' column
print("Null check:")
print(equity_features_df["is_quality_screen_pass"].isna().sum())  # Prints count of null values

# Displaying the count of True/False values to see how many companies pass or fail the quality screen
print("\nPass / Fail counts:")
print(equity_features_df["is_quality_screen_pass"].value_counts())  # Prints pass/fail counts

# Sample view of the dataframe showing a preview of the first 10 rows for selected columns
print("\nSample view:")
# This will show a sample of 10 rows displaying:
# - company_name: Name of the company
# - profitability_bucket: The profitability classification (e.g., 'strong', 'weak')
# - capital_efficiency_flag: Whether the company has acceptable capital efficiency
# - is_quality_screen_pass: The result of the quality screen check (True/False)
equity_features_df[
    [
        "company_name",  # Company name for identification
        "profitability_bucket",  # Profitability bucket to see if it's "strong"
        "capital_efficiency_flag",  # Capital efficiency flag to see if it's acceptable
        "is_quality_screen_pass",  # Whether the company passes the quality screen
    ]
].head(10) # Displays the first 10 rows

Null check:
0

Pass / Fail counts:
is_quality_screen_pass
False    2915
True      230
Name: count, dtype: int64

Sample view:


,company_name,profitability_bucket,capital_efficiency_flag,is_quality_screen_pass
0,Samvardh. Mothe.,weak,False,False
1,Bosch,moderate,True,False
2,Uno Minda,moderate,True,False
3,Bharat Forge,moderate,False,False
4,MRF,moderate,False,False
5,Schaeffler India,moderate,True,False
6,Tube Investments,weak,True,False
7,Balkrishna Inds,moderate,True,False
8,Endurance Tech.,moderate,True,False
9,Apollo Tyres,weak,False,False


#### **Screening Logic — Growth Screen**

In [20]:
# -------------------------------------------
# Screening Logic — Growth Screen
# -------------------------------------------

# Business rule:
# A company passes the growth screen if:
# - The growth signal is positive (i.e., is_growth_company is True)
# - The growth bucket is "strong"

# The following line creates a new column 'is_growth_screen_pass' that evaluates to True 
# if both conditions are met for each company in the dataframe:
# 1. 'is_growth_company' must be True (positive growth signal)
# 2. 'growth_bucket' must be "strong"
equity_features_df["is_growth_screen_pass"] = (
    equity_features_df["is_growth_company"]  # Check if the company has a positive growth signal
    & equity_features_df["growth_bucket"].eq("strong")  # Check if the growth bucket is 'strong'
)

# -------------------------------------------
# Validations
# -------------------------------------------

# Checking for any missing (null) values in the 'is_growth_screen_pass' column
print("Null check:")
print(equity_features_df["is_growth_screen_pass"].isna().sum())  # Prints count of null values

# Displaying the count of True/False values to show how many companies pass or fail the growth screen
print("\nPass / Fail counts:")
print(equity_features_df["is_growth_screen_pass"].value_counts())  # Prints pass/fail counts

# Sample view of the dataframe showing the first 10 rows for selected columns
print("\nSample view:")
# This will show a sample of 10 rows displaying:
# - company_name: Name of the company
# - is_growth_company: Whether the company has a positive growth signal (True/False)
# - growth_bucket: The growth classification (e.g., 'strong', 'weak')
# - is_growth_screen_pass: The result of the growth screen check (True/False)
equity_features_df[
    [
        "company_name",  # Company name for identification
        "is_growth_company",  # Whether the company has a positive growth signal
        "growth_bucket",  # Growth bucket to see if it's "strong"
        "is_growth_screen_pass",  # Whether the company passes the growth screen
    ]
].head(10)  # Displays the first 10 rows


Null check:
0

Pass / Fail counts:
is_growth_screen_pass
False    2729
True      416
Name: count, dtype: int64

Sample view:


,company_name,is_growth_company,growth_bucket,is_growth_screen_pass
0,Samvardh. Mothe.,False,weak,False
1,Bosch,True,moderate,False
2,Uno Minda,True,moderate,False
3,Bharat Forge,False,weak,False
4,MRF,False,weak,False
5,Schaeffler India,True,moderate,False
6,Tube Investments,False,weak,False
7,Balkrishna Inds,False,weak,False
8,Endurance Tech.,True,moderate,False
9,Apollo Tyres,False,weak,False


#### **Persist Screening Outputs**

In [21]:
# -------------------------------------------
# Persist Screening Outputs
# -------------------------------------------

# Importing Path class from pathlib to handle file and directory paths
from pathlib import Path

# Defining the directory where processed files will be saved
# 'data/processed' will be the parent directory for the output files
processed_dir = Path("data") / "processed"

# Create the 'processed' directory if it doesn't already exist (with parents if needed)
processed_dir.mkdir(parents=True, exist_ok=True)

# 1) Saving the full dataset with screening flags to a CSV file
# This will include all the companies along with the results of the various screening flags
full_screen_path = processed_dir / "equity_screened_full.csv"
equity_features_df.to_csv(full_screen_path, index=False)  # Write the full dataframe to a CSV

# 2) Saving only the companies that passed the basic screen to a CSV file
# Filter the dataframe to include only rows where 'is_basic_screen_pass' is True
basic_screen_df = equity_features_df[equity_features_df["is_basic_screen_pass"]]
basic_screen_path = processed_dir / "equity_basic_screen.csv"
basic_screen_df.to_csv(basic_screen_path, index=False)  # Write the filtered dataframe to a CSV

# 3) Saving only the companies that passed the quality screen to a CSV file
# Filter the dataframe to include only rows where 'is_quality_screen_pass' is True
quality_screen_df = equity_features_df[equity_features_df["is_quality_screen_pass"]]
quality_screen_path = processed_dir / "equity_quality_screen.csv"
quality_screen_df.to_csv(quality_screen_path, index=False)  # Write the filtered dataframe to a CSV

# 4) Saving only the companies that passed the growth screen to a CSV file
# Filter the dataframe to include only rows where 'is_growth_screen_pass' is True
growth_screen_df = equity_features_df[equity_features_df["is_growth_screen_pass"]]
growth_screen_path = processed_dir / "equity_growth_screen.csv"
growth_screen_df.to_csv(growth_screen_path, index=False)  # Write the filtered dataframe to a CSV

# Printing the paths of the saved files for reference
print("Saved files:")
print(full_screen_path)  # Path to the full dataset with all screening results
print(basic_screen_path)  # Path to the filtered dataset for companies passing the basic screen
print(quality_screen_path)  # Path to the filtered dataset for companies passing the quality screen
print(growth_screen_path)  # Path to the filtered dataset for companies passing the growth screen

# Printing the row counts for each of the datasets to show how many companies passed each screen
print("\nRow counts:")
print("Full:", equity_features_df.shape[0])  # Total number of rows in the full dataset
print("Basic Screen:", basic_screen_df.shape[0])  # Number of companies passing the basic screen
print("Quality Screen:", quality_screen_df.shape[0])  # Number of companies passing the quality screen
print("Growth Screen:", growth_screen_df.shape[0])  # Number of companies passing the growth screen

Saved files:
data\processed\equity_screened_full.csv
data\processed\equity_basic_screen.csv
data\processed\equity_quality_screen.csv
data\processed\equity_growth_screen.csv

Row counts:
Full: 3145
Basic Screen: 821
Quality Screen: 230
Growth Screen: 416
